Data about unemployment on municipality level are loaded from https://www.regionalstatistik.de/genesis//online?operation=table&code=13211-01-03-5&bypass=true&levelindex=1&levelid=1762338172400#abreadcrumb

Data contain ID column as identifier.

ID consist of:
- Land (2 digits) which represents Bundesland
- Regierungsbezirk (1 digit) which represents governmental district
- Kreis (2 digits) which represents district
- Gemeinde (3 digits) which represents municipality

There are also rows representing higher administrative levels (e.g. districts) which can be identified by having just two or five digits in ID column.
We cannot just filter them out because Hamburg has no districts and its municipalities have just two digits in ID column.

In [ ]:
# load municipalities for filtering and merging
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

muni_data_path = "../../data/raw/municipalities_2022.csv"
municipalities = MunicipalityFeature(muni_data_path)
municipalities = municipalities.load_transform()
municipalities.head()

## Load of unemployment data

In [ ]:
from geoscore_de.data_flow.features.unemployment import UnemploymentFeature

df = UnemploymentFeature("../../data/raw/features/13211-01-03-5-unemployment-2025.csv").load()

In [ ]:
df.dtypes

In [ ]:
df_merged = municipalities.merge(df, on="AGS", how="outer", indicator=True)
df_merged.head()

In [ ]:
df_merged["_merge"].value_counts()

All municipalities from 2022 municipality data are present in unemployment data. There are extra rows in unemployment data which represent higher administrative levels. Same in most of the data from regionalstatistik.de.

In [ ]:
df_merged = municipalities.merge(df, on="AGS", how="left")

In [ ]:
# get missing per statistics
statistics = ["total", "foreigners", "disabled", "age_15_20", "age_15_25", "age_55_65", "long_term"]
for stat in statistics:
    missing_size = df_merged[df_merged[stat].isna()].shape[0]
    print(f"Number of municipalities with missing {stat} data: {missing_size}")

Problem is not with merging the data, but with the data itself. Missing data are also predictors which can be used by the model.
It can represent that the municipality is very small and thus not included in unemployment data or that the municipality statistics office did not report the data for some reason. In both cases, it can be useful for the model to know that the data is missing.
We can see that most of the missing data are in the age groups 15-20 and 15-25. Least missing data are in the total unemployment rate, age group 55-65 and long-term unemployment rate.
We will focus on those three statistics in the analysis and we will keep the missing data as they are.

## Data exploration

In [ ]:
from geoscore_de.data_flow.features.unemployment import UnemploymentFeature

df = UnemploymentFeature(
    "../../data/raw/features/13211-01-03-5-unemployment-2025.csv",
    tform_data_path="../../data/tform/features/unemployment_2025.csv",
    municipality_data_path=muni_data_path,
).load_transform()
df_merged = municipalities.merge(df, on="AGS", how="left")
df_merged.head()

We have multiple columns with unemployment data. Total unemployment, unemployment of foreigners, unemployment of disabled people, unemployment per age groups 15-20, 15-25, 55-65 and long term unemployment. All of these columns are normalized by population of the municipality.

In [ ]:
import plotnine as gg

# show histogram of unemployment rate
(
    gg.ggplot(df_merged, gg.aes(x="total"))
    + gg.geom_histogram(binwidth=0.001)
    + gg.scale_x_continuous(labels=lambda labels: ["{:.0f}%".format(x * 100) for x in labels])
    + gg.labs(
        title="Distribution of Unemployment Rate in German Municipalities",
        x="Unemployment Rate",
        y="Count of Municipalities",
    )
)

In [ ]:
# now show it on map of germany
from geoscore_de.data_flow.geo import load_geo_data

gdf = load_geo_data("../../data/georef-germany-gemeinde.csv")

In [ ]:
# join data by ags
gdf_merged = gdf.merge(df_merged, on="AGS", how="outer", indicator=True)

In [ ]:
# print count of missing values
gdf_merged["_merge"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

gdf_merged.plot(
    column="total",
    legend=True,
    cmap="YlOrRd",
    figsize=(12, 8),
    vmin=0,
    vmax=0.06,
    missing_kwds={"color": "gray"},
)
plt.title("Total Unemployment Rate in Germany")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

gdf_merged.plot(
    column="age_55_65",
    legend=True,
    cmap="YlOrRd",
    figsize=(12, 8),
    vmin=0,
    vmax=0.020,
    missing_kwds={"color": "gray"},
)
plt.title("Unemployment Rate of Age Group 55-65 in Germany")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

gdf_merged.plot(
    column="long_term",
    legend=True,
    cmap="YlOrRd",
    figsize=(12, 8),
    vmin=0,
    vmax=0.03,
    missing_kwds={"color": "gray"},
)
plt.title("Long-term Unemployment Rate in Germany")
plt.show()

On all three graphs, we can see that unemployment is higher in the post eastern Germany and in North Rhine-Westphalia. Espetially visible in age group 55-65 and long-term unemployment. Age group can be explained by the fact that in the eastern Germany, there are more older people and thus higher unemployment in this age group per population.

## Compare year 2025 with 2020
Now we will compare unemployment data from 2025 with data from 2020. 
We will subtract 2020 data from 2025 data to get the change in unemployment.

In [ ]:
from geoscore_de.data_flow.features.unemployment import load_unemployment_data

df_2020 = load_unemployment_data("../../data/raw/features/13211-01-03-5-unemployment-2020.csv")
df_2020

In [ ]:
mmerged = gdf_merged.merge(df_2020, on="AGS", suffixes=("_2025", "_2020"), how="left")
mmerged["unemployment_rate_2020"] = mmerged["total_2020"] / mmerged["Persons"]

mmerged.columns

In [ ]:
mmerged["yty_unemployment_rate"] = mmerged["total_2025"] - mmerged["unemployment_rate_2020"]
mmerged[["AGS", "total_2025", "unemployment_rate_2020", "yty_unemployment_rate"]]

In [ ]:
import plotnine as gg

# show histogram of unemployment rate
(
    gg.ggplot(mmerged, gg.aes(x="yty_unemployment_rate"))
    + gg.geom_histogram(binwidth=0.001)
    + gg.scale_x_continuous(labels=lambda labels: ["{:.0f}%".format(x * 100) for x in labels])
    + gg.labs(
        title="Change in Unemployment Rate from 2020 to 2025 in German Municipalities",
        x="Unemployment Rate",
        y="Count of Municipalities",
    )
)

In [ ]:
import matplotlib.pyplot as plt

mmerged.plot(
    column="yty_unemployment_rate",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=-0.025,
    vmax=0.025,
    missing_kwds={"color": "gray"},
)
plt.title("Change in Unemployment Rate from 2020 to 2025 in Germany")
plt.show()

Based on the change in unemployment map, we can see positive change in most of the eastern Germany. This can be explained by the fact that unemployment in eastern Germany is higher and thus there is more room for improvement. In western Germany, the unemployment rate stagnates or even increases a bit. This shows that the gap between eastern and western Germany is slowly closing.